# GMM 与 EM 如何实现软聚类？

**面试回答：**GMM 用混合高斯表达多个潜在人群；E 步算责任度，M 步按责任度更新权重、均值和方差。责任度是模型下的软归属，不等于业务真相。

## 真实案例

10 张客服工单的处理分钟数混合了简单咨询和复杂投诉两种流程，边界工单应保留不确定性。

In [1]:
import numpy as np  # 导入 NumPy 手写一维 GMM。
ticket=np.array(['T01','T02','T03','T04','T05','T06','T07','T08','T09','T10'])  # 构造工单编号。
x=np.array([4.,6.,7.,9.,11.,24.,28.,30.,35.,18.])  # 记录处理分钟数。
print('工单 | 处理分钟')  # 输出原始工单表头。
for n,v in zip(ticket,x):  # 逐条展示业务样本。
    print(f'{n} | {v:6.1f}')  # 输出一条工单。

工单 | 处理分钟
T01 |    4.0
T02 |    6.0
T03 |    7.0
T04 |    9.0
T05 |   11.0
T06 |   24.0
T07 |   28.0
T08 |   30.0
T09 |   35.0
T10 |   18.0


## Baseline / 基线

硬阈值 15 分钟把工单切为两类，无法表达 18 分钟边界工单的不确定性。

In [2]:
baseline=(x>=15).astype(int)  # 用固定时长阈值生成硬分配。
print('硬阈值簇:',baseline.tolist())  # 输出基线分类。
print('T10 被硬归到复杂类，基线没有任何置信度。')  # 指出基线缺少软信息。

硬阈值簇: [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]
T10 被硬归到复杂类，基线没有任何置信度。


In [3]:
def normal_pdf(value,mean,var):  # 定义一维高斯密度函数。
    return np.exp(-0.5*(value-mean)**2/var)/np.sqrt(2*np.pi*var)  # 返回高斯概率密度。
weight=np.array([.5,.5])  # 初始化两个混合分量权重。
mean=np.array([7.,29.])  # 初始化简单与复杂流程均值。
var=np.array([20.,30.])  # 初始化两个流程方差。
log_likelihood=[]  # 保存每轮对数似然。
for step in range(20):  # 交替执行 E 步和 M 步。
    density=np.column_stack([weight[k]*normal_pdf(x,mean[k],var[k]) for k in range(2)])  # 计算样本属于每分量的未归一化概率。
    responsibility=density/density.sum(axis=1,keepdims=True)  # E 步归一化得到软责任度。
    nk=responsibility.sum(axis=0)  # 统计各分量的有效样本数。
    weight=nk/len(x)  # M 步更新混合权重。
    mean=(responsibility*x[:,None]).sum(axis=0)/nk  # M 步更新加权均值。
    var=(responsibility*(x[:,None]-mean)**2).sum(axis=0)/nk+1e-3  # M 步更新方差并加下界。
    log_likelihood.append(float(np.log(density.sum(axis=1)).sum()))  # 记录更新前密度的对数似然。
print('责任度前五行:',np.round(responsibility[:5],3))  # 输出软分配中间量。
print('均值/方差:',np.round(mean,2),np.round(var,2))  # 输出最终参数。
print('对数似然尾部:',np.round(log_likelihood[-4:],2))  # 输出 EM 收敛轨迹。

责任度前五行: [[0.999 0.001]
 [0.999 0.001]
 [0.999 0.001]
 [0.995 0.005]
 [0.967 0.033]]
均值/方差: [ 7.38 26.86] [ 5.81 34.81]
对数似然尾部: [-34.22 -34.22 -34.22 -34.22]


## 结果解读

E 步让边界工单在两个流程之间分摊责任，M 步再以责任度加权更新参数；方差下界防止单样本分量坍缩。

In [4]:
print('工单 | 分钟 | 简单责任度 | 复杂责任度')  # 输出结果表头。
for n,v,r in zip(ticket,x,responsibility):  # 逐条展示软归属。
    print(f'{n} | {v:4.0f} | {r[0]:10.3f} | {r[1]:10.3f}')  # 输出每张工单责任度。
print('生产差距：需用对数域计算、多初始化、分量数选择与数据漂移监控。')  # 说明工程能力缺口。

工单 | 分钟 | 简单责任度 | 复杂责任度
T01 |    4 |      0.999 |      0.001
T02 |    6 |      0.999 |      0.001
T03 |    7 |      0.999 |      0.001
T04 |    9 |      0.995 |      0.005
T05 |   11 |      0.967 |      0.033
T06 |   24 |      0.000 |      1.000
T07 |   28 |      0.000 |      1.000
T08 |   30 |      0.000 |      1.000
T09 |   35 |      0.000 |      1.000
T10 |   18 |      0.000 |      1.000
生产差距：需用对数域计算、多初始化、分量数选择与数据漂移监控。


## 失败案例与修复

若方差允许为零，EM 会把一个分量缩到单点，密度虚高。修复是方差下界和最小有效样本门槛。

In [5]:
tiny_var=1e-12  # 构造近零方差失败条件。
unstable_density=normal_pdf(np.array([18.]),18.,tiny_var)[0]  # 计算近零方差产生的虚高密度。
stable_density=normal_pdf(np.array([18.]),18.,1e-3)[0]  # 计算方差下界后的有限密度。
print('失败：近零方差密度=',f'{unstable_density:.2e}')  # 输出坍缩信号。
print('修复：方差下界密度=',f'{stable_density:.2e}')  # 输出稳定后密度。
print('注意：密度可大于一，它不是离散概率。')  # 澄清概率密度概念。

失败：近零方差密度= 3.99e+05
修复：方差下界密度= 1.26e+01
注意：密度可大于一，它不是离散概率。


In [6]:
assert len(ticket)>=5  # 保护案例包含足够工单。
assert np.allclose(responsibility.sum(axis=1),1.0)  # 保护每条工单责任度归一化。
assert log_likelihood[-1]>=log_likelihood[0]  # 保护 EM 似然未下降。
assert np.all(var>0)  # 保护方差下界生效。